# Week 13 Lab: k-means by Hand

You'll write the two steps at the heart of k-means yourself, then (optionally) use the adjusted Rand index to
see how stable a clustering is as k changes.

**Core**: Exercise 1 (about 40 minutes). **Extension**: Exercise 2 (optional).

## Exercise 1: Implementing k-means

Most of the code is provided. You implement the two key calculations: assigning points to clusters and updating
the centroids.

### Setup
Run this first: it imports the libraries and creates the sample data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Generate sample data
np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

# Plot the data
plt.scatter(X[:, 0], X[:, 1], s=50)
plt.title("Generated Data")
plt.show()

### Task 1: Initialize centroids
The initialization code is provided. It picks k random points as the starting centroids:

In [ ]:
def initialize_centroids(X, k):
    n_samples = X.shape[0]
    random_indices = np.random.choice(n_samples, k, replace=False)
    centroids = X[random_indices]
    return centroids

# Initialize with k=4 clusters
k = 4
centroids = initialize_centroids(X, k)

### Task 2: Implement Cluster Assignment
Fill in the missing code in the `assign_clusters` function. This function should:
1. Calculate the distance between each point and each centroid
2. Assign each point to the nearest centroid

In [ ]:
def assign_clusters(X, centroids):
    # Calculate distances between each point and each centroid
    # Hint: X[:, np.newaxis] - centroids has shape (n_points, k, n_features);
    # square it, sum over the last axis (axis=2), and take the square root
    distances = # YOUR CODE HERE
    
    # Find the closest centroid for each point
    # Hint: Use np.argmin with axis=1
    labels = # YOUR CODE HERE
    
    return labels

### Task 3: Implement Centroid Updates
Fill in the missing code in the `update_centroids` function. This function should:
1. Calculate the mean position of all points assigned to each cluster
2. Return these means as the new centroids

In [ ]:
def update_centroids(X, labels, k):
    centroids = np.zeros((k, X.shape[1]))
    for i in range(k):
        # Calculate mean position of points in cluster i
        # Hint: Use boolean indexing with labels == i
        centroids[i] = # YOUR CODE HERE
    return centroids

### Task 4: Run K-means
The main k-means loop is provided. This will use your functions to run the algorithm:

In [ ]:
def kmeans(X, k, max_iters=100):
    # Initialize centroids
    centroids = initialize_centroids(X, k)
    
    for _ in range(max_iters):
        # Assign clusters
        old_labels = assign_clusters(X, centroids)
        
        # Update centroids
        centroids = update_centroids(X, old_labels, k)
        
        # Check for convergence
        new_labels = assign_clusters(X, centroids)
        if np.all(old_labels == new_labels):
            break
            
    return centroids, new_labels

# Run kmeans
final_centroids, labels = kmeans(X, k)

### Task 5: Visualize Results
Use this code to visualize your clustering results:

In [ ]:
plt.figure(figsize=(10, 4))

# Plot original data
plt.subplot(121)
plt.scatter(X[:, 0], X[:, 1], s=50)
plt.title("Original Data")

# Plot clustered data
plt.subplot(122)
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=50)
plt.scatter(final_centroids[:, 0], final_centroids[:, 1], 
            c='red', marker='x', s=200, linewidths=3)
plt.title("Clustered Data")
plt.show()

### Discussion questions
1. How well did your implementation work? Are the clusters what you expected?
2. What happens if you change the number of clusters (k)?
3. Try changing the `cluster_std` parameter in `make_blobs`. How does this affect the clustering?
4. Run the k-means cell several times. Do you always get the same clusters? Why or why not?

---

## Exercise 2 (extension): How stable is the clustering?

The lecture describes *stability analysis*: checking whether a clustering stays the same when something changes.
Here the thing that changes is k-means' random starting centroids. For each k you run k-means several times with
different `random_state` values and `n_init=1` (one start per run, so the runs really can differ), then compare
each run with the previous one using the **adjusted Rand index** (ARI: 1 = identical groupings, about 0 = no more
alike than random). A k whose runs keep agreeing is *stable*. Both data sets below have four true clusters; one is
clear and one is ambiguous.

### Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.datasets import make_blobs

# Two data sets with the same four centers: one clear, one ambiguous
X_clear, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=1)
X_ambig, _ = make_blobs(n_samples=300, centers=4, cluster_std=2.5, random_state=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(X_clear[:, 0], X_clear[:, 1], s=15)
ax1.set_title("Clear clusters")
ax2.scatter(X_ambig[:, 0], X_ambig[:, 1], s=15)
ax2.set_title("Ambiguous clusters")
plt.show()

### Task 1: Measure stability for each k
Complete the function so it returns the mean ARI between consecutive runs for every k:

In [ ]:
def clustering_stability(X, k_range, n_runs=10):
    """For each k, run k-means n_runs times with different seeds; return the mean ARI between consecutive runs."""
    stability_scores = []
    for k in k_range:
        ari_scores = []
        prev_labels = None
        for seed in range(n_runs):
            # Hint: KMeans(n_clusters=k, n_init=1, random_state=seed)
            labels = # YOUR CODE HERE

            if prev_labels is not None:
                # Hint: use adjusted_rand_score
                ari = # YOUR CODE HERE
                ari_scores.append(ari)
            prev_labels = labels
        stability_scores.append(np.mean(ari_scores))
    return stability_scores

k_range = range(2, 9)
stability_clear = clustering_stability(X_clear, k_range)
stability_ambig = clustering_stability(X_ambig, k_range)
for k, c, a in zip(k_range, stability_clear, stability_ambig):
    print(f"k={k}: clear {c:.2f}   ambiguous {a:.2f}")

### Task 2: Plot the stability scores
Complete the function to plot both data sets' scores against k on one chart:

In [ ]:
def plot_stability_comparison(k_range, stability_clear, stability_ambig):
    plt.figure(figsize=(10, 6))

    # Plot stability scores against k
    # Hint: plt.plot(list(k_range), ..., marker='o', label=...)
    plt.plot(# YOUR CODE HERE)  # Clear clusters
    plt.plot(# YOUR CODE HERE)  # Ambiguous clusters

    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Mean ARI between runs')
    plt.title('Clustering stability')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_stability_comparison(k_range, stability_clear, stability_ambig)

### Task 3: Look at the clusterings
Complete the scatter call to see the clusters for each k:

In [ ]:
def plot_clustering_progression(X, k_range):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for idx, k in enumerate(list(k_range)[:6]):  # Plot first 6 k values
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X)
        
        # Create scatter plot
        # Hint: Use different colors for different clusters
        axes[idx].scatter(# YOUR CODE HERE)
        axes[idx].set_title(f'k={k}')
    
    plt.tight_layout()
    plt.show()

# Visualize clustering progression for both datasets
plot_clustering_progression(X_clear, k_range)
plot_clustering_progression(X_ambig, k_range)

### Discussion questions
1. For the clear data, which values of k are stable, and where does stability drop? What does the clustering plot
   show happening at that k?
2. How does the ambiguous data differ? What does that tell you about its clusters?
3. Stability is not the same as correctness: give an example of a k that is stable but not the "right" number of
   clusters.